In [18]:
import HTSeq
import numpy as np
np.random.seed(42)
import pysam
import os

from price2.reference_annotation import ReferenceAnnotation
from price2.rgr_finder import RGRFinder
from price2.genomic_region import GenomicRegion
from price2.cleavage_model import CleavageModel


In [19]:
pl = np.loadtxt('/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/upstream_cleavage_model')
pl = pl / pl.sum()
pr = np.loadtxt('/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/downstream_cleavage_model')
pr = pr / pr.sum()
u = .1
cm = CleavageModel(pl, pr, u)

In [20]:
reference_gtf = '/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/orf_deconvolution.gtf'
ra = ReferenceAnnotation(reference_gtf)
#ra = ReferenceAnnotation('/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/test/test.gtf')
ra.transcripts

{'transcript1': <price2.genomic_features.Transcript at 0x7fe914e13280>,
 'transcript2': <price2.genomic_features.Transcript at 0x7fe914e13700>,
 'transcript3': <price2.genomic_features.Transcript at 0x7fe9146945e0>,
 'transcript4': <price2.genomic_features.Transcript at 0x7fe914694ac0>,
 'transcript5': <price2.genomic_features.Transcript at 0x7fe914554220>,
 'transcript6': <price2.genomic_features.Transcript at 0x7fe9145578b0>,
 'transcript7': <price2.genomic_features.Transcript at 0x7fe914576c80>,
 'transcript8': <price2.genomic_features.Transcript at 0x7fe914575690>}

In [21]:
seqs = {}

fasta_1 = 'N'*7000


orf_gtf = '/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/orf_deconvolution_ORFs.gtf'

final_orfs = {}
for orf in HTSeq.GFF_Reader(orf_gtf):
    if orf.iv.chrom == 'chr1':
        if orf.attr['ID'] not in final_orfs:
            if orf.iv.strand == '+':
                fasta_1 = fasta_1[:orf.iv.start] + 'ATG' + fasta_1[orf.iv.start+3:]
            else:
                fasta_1 = fasta_1[:orf.iv.end-3] + 'CAT' + fasta_1[orf.iv.end:]
        final_orfs[orf.attr['ID']] = orf

for orf in final_orfs.values():
    if orf.iv.strand == '+':
        fasta_1 = fasta_1[:orf.iv.end-3] + 'TAG' + fasta_1[orf.iv.end:]
    else:
        fasta_1 = fasta_1[:orf.iv.start-3] + 'CTA' + fasta_1[orf.iv.start:]

seqs['chr1'] = HTSeq.Sequence(fasta_1.encode('ascii'), 'chr1')

fasta_2 = 'N'*10000

orf_gtf = '/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/orf_deconvolution_ORFs.gtf'

final_orfs = {}
for orf in HTSeq.GFF_Reader(orf_gtf):
    if orf.iv.chrom == 'chr2':
        if orf.attr['ID'] not in final_orfs:
            if orf.iv.strand == '+':
                fasta_2 = fasta_2[:orf.iv.start] + 'ATG' + fasta_2[orf.iv.start+3:]
            else:
                fasta_2 = fasta_2[:orf.iv.end-3] + 'CAT' + fasta_2[orf.iv.end:]
        final_orfs[orf.attr['ID']] = orf

for orf in final_orfs.values():
    if orf.iv.strand == '+':
        fasta_2 = fasta_2[:orf.iv.end-3] + 'TAG' + fasta_2[orf.iv.end:]
    else:
        fasta_2 = fasta_2[:orf.iv.start] + 'CTA' + fasta_2[orf.iv.start+3:]

seqs['chr2'] = HTSeq.Sequence(fasta_2.encode('ascii'), 'chr2')


with open('/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/orf_deconvolution.fasta', 'w') as f:
    for seq in seqs.values():
        seq.write_to_fasta_file(f)

In [22]:
of = RGRFinder(ra.transcripts, seqs)


In [23]:
class ReadGeneratingRegion:
    id: str
    type: str
    genomic_region: GenomicRegion
    activity: float
    length: int

    def __init__(self, activity: float, genomic_region, id: str, type: str):
        self.type = type
        self.id = id
        self.genomic_region = genomic_region
        self.activity = activity

read_generating_regions = {}

for orf in HTSeq.GFF_Reader(orf_gtf):
    if not orf.attr['ID'] in read_generating_regions:
        read_generating_regions[orf.attr['ID']] = ReadGeneratingRegion(
            int(orf.attr['activity']),
            GenomicRegion([], orf.iv.chrom, orf.iv.strand),
            id = orf.attr['ID'],
            type = 'ORF'
            )

    read_generating_regions[orf.attr['ID']].genomic_region.add_interval(orf.iv)

for region in HTSeq.GFF_Reader(reference_gtf):
    if region.type != 'transcript':
        continue
    if not region.attr['transcript_id'] in read_generating_regions:
        read_generating_regions[region.attr['transcript_id']] = ReadGeneratingRegion(
            int(region.attr['activity']),
            GenomicRegion([], region.iv.chrom, region.iv.strand),
            id = region.attr['transcript_id'],
            type = 'NOISE'
            )
    read_generating_regions[region.attr['transcript_id']].genomic_region.add_interval(region.iv)


for region in read_generating_regions.values():
    region.length = len(region.genomic_region)
    region.read_likelihood = region.length * region.activity

read_generating_regions = list(read_generating_regions.values())
probabilities = [orf.read_likelihood for orf in read_generating_regions]
probabilities = probabilities / np.sum(probabilities)


In [24]:
def generate_read():
    region = np.random.choice(read_generating_regions, p=probabilities)
    l,r,u = cm.rvs()
    l = l[0]
    r = r[0]
    u = u[0]
    if u:
        u_observed = np.random.choice([True, False], p=[.75, .25])
    else:
        u_observed = False
    if region.type == 'ORF':
        p_site_position_in_orf = np.random.randint(0, region.length//3)*3
    elif region.type == 'NOISE':
        p_site_position_in_orf = np.random.randint(0, region.length)
    read_interval_on_orf = (p_site_position_in_orf - l - int(u), p_site_position_in_orf + r + 3)
    read_interval_on_genome = region.genomic_region.map(read_interval_on_orf)
    frame = (-l-int(u))%3
    return read_interval_on_genome, u_observed, region.id, l + r + 3 + int(u), frame

def generate_sam(read_num: int, directory: str, id: str):
    with open(f'{directory}/{id}.sam', 'w') as f:
        f.write('@HD	VN:1.6	SO:coordinate')
        f.write('\n@SQ	SN:chr1	LN:7000')
        f.write('\n@SQ	SN:chr2	LN:7000')
        for i in range(read_num):
            read_gr, u_observed, orf_id, read_length, frame = generate_read()
            read_id = f'r{i:08d}'
            if read_gr.strand == '+':
                bit_flag = 0
            else:
                bit_flag = 16
            chrom = read_gr.chrom
            start = read_gr.intervals[0].start + 1
            mapq = 0

            cigar = ''
            ivs = read_gr.intervals
            if read_gr.strand == '-':
                ivs = ivs[::-1]
            
            for c, iv in enumerate(read_gr.intervals):
                if c>0:
                    cigar += f'{iv.start - read_gr.intervals[c-1].end}N'
                cigar += f'{iv.end - iv.start}M'
            if u_observed and read_gr.strand == '+':
                cigar = f'1S{cigar}'
            elif u_observed and read_gr.strand == '-':
                cigar = f'{cigar}1S'

            r_next = '*'
            p_next = 0
            tlen = 0
            seq = 'N'*(read_length+int(u_observed))
            qual = '*'
            attributes = f'NH:i:1\tMD:Z:{read_length}\tCO:Z:{orf_id}\tXf:f:{frame}'

            f.write(f'\n{read_id}\t{bit_flag}\t{chrom}\t{start}\t{mapq}\t{cigar}\t{r_next}\t{p_next}\t{tlen}\t{seq}\t{qual}\t{attributes}')
    
    open(f'{directory}/{id}.unsorted.bam', 'w').close()
    pysam.view('-bS', f'{directory}/{id}.sam', '-o', f'{directory}/{id}.unsorted.bam', save_stdout=f'{directory}/{id}.unsorted.bam')
    open(f'{directory}/{id}.bam', 'w').close()
    pysam.sort(f'{directory}/{id}.unsorted.bam', '-o', f'{directory}/{id}.bam', save_stdout=f'{directory}/{id}.bam')
    os.remove(f'{directory}/{id}.unsorted.bam')
    pysam.index(f'{directory}/{id}.bam')

In [25]:
#generate_sam(100_000, '/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/runs', 'SRR1')
#generate_sam(1_000_000, '/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/runs', 'SRR2')

generate_sam(10_000, '/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/runs_simple', 'SRR1')
generate_sam(1_000, '/projects/viro/maidhof/price2/orf_deconvolution_development/data_advanced/runs_simple', 'SRR2')
